In [1]:
import torch
torch.__version__

/Users/abhideb/miniconda3/envs/pytorch_env/lib/python3.12/site-packages/torch/nn/modules/transformer.py:20: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  device: torch.device = torch.device(torch._C._get_default_device()),  # torch.device('cpu'),


'2.2.2'

In [2]:
# Is GPU available?
torch.cuda.is_available()

False

In [3]:
%load_ext watermark
%watermark -a "Abhitesh Debnath" -v -p torch,numpy,transformers -m -g

Author: Abhitesh Debnath

Python implementation: CPython
Python version       : 3.12.11
IPython version      : 9.1.0

torch       : 2.2.2
numpy       : unknown
transformers: unknown

Compiler    : Clang 14.0.6 
OS          : Darwin
Release     : 23.6.0
Machine     : x86_64
Processor   : i386
CPU cores   : 4
Architecture: 64bit

Git hash: e2a61b3b1d5f2c32b590532397ea0024e139607a



## Creating PyTorch Tensors

In [4]:
tensor0d = torch.tensor(1)
tensor0d

tensor(1)

In [5]:
tensor1d = torch.tensor([1, 2])
tensor1d

tensor([1, 2])

In [6]:
tensor2d = torch.tensor([[1, 2, 3],
                         [2, 3, 4]])
tensor2d

tensor([[1, 2, 3],
        [2, 3, 4]])

In [7]:
tensor3d = torch.tensor([[[1, 2], [3, 4]],
                         [[5, 6], [7, 8]]])
tensor3d

tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])

In [8]:
tensor1d.dtype

torch.int64

In [9]:
floatvec = tensor1d.to(torch.float32)
print(floatvec.dtype)
floatvec

torch.float32


tensor([1., 2.])

In [10]:
tensor2d.shape

torch.Size([2, 3])

In [11]:
tensor2d.view([3, 2])

tensor([[1, 2],
        [3, 2],
        [3, 4]])

In [12]:
tensor2d.T

tensor([[1, 2],
        [2, 3],
        [3, 4]])

In [13]:
tensor2d.matmul(tensor2d.T)

tensor([[14, 20],
        [20, 29]])

In [14]:
tensor2d @ tensor2d.T

tensor([[14, 20],
        [20, 29]])

## A Logistic Regression Forward pass

In [15]:
import torch.nn.functional as F
y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2])
b = torch.tensor([0.0])
z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)
loss

tensor(0.0852)

## Computing gradients via autograd

In [16]:
import torch.nn.functional as F
from torch.autograd import grad

In [17]:
y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

In [18]:
print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [19]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


## A multilayer perceptron with two hidden layers

In [ ]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):  # Coding the number of inputs and outputs as variables allows us to reuse the same code for datasets with diff number of features and classes
        super().__init__()

        self.layers = torch.nn.Sequential(        # Defining layers as a modified Sequential function
            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),      # The Linear layer takes the number of input and output nodes as arguments
            torch.nn.ReLU(),                      # Nonlinear activation functions are placed between the hidden layers.

            # 2nd hidden layer
            torch.nn.Linear(30, 20),              # Number of output nodes of one hidden layer has to match the number of inputs of the next layer
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs)
        )

    def forward(self, x):
        logits = self.layers(x)                   # The output of last layer is called logits. Calling layers() fn defined earlier
        return logits

In [27]:
torch.manual_seed(42)
model = NeuralNetwork(50, 3)

In [28]:
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [29]:
num_parameters = sum([p.numel() for p in model.parameters() if p.requires_grad])
print(f"The number of parameters in the model is {num_parameters}")

The number of parameters in the model is 2213


In [30]:
print(model.layers[0].weight)

Parameter containing:
tensor([[ 0.1081,  0.1174, -0.0331,  ...,  0.0253,  0.0718, -0.0862],
        [-0.1400, -0.0546, -0.1085,  ..., -0.0477, -0.0501, -0.1368],
        [-0.0810,  0.0353, -0.0187,  ...,  0.1142,  0.1288, -0.1121],
        ...,
        [-0.0031, -0.0573,  0.0515,  ...,  0.0271, -0.0928, -0.1175],
        [-0.0444, -0.1318, -0.0660,  ...,  0.0647, -0.1230, -0.0531],
        [ 0.0023, -0.1223,  0.0797,  ...,  0.0369,  0.0862,  0.1328]],
       requires_grad=True)


In [31]:
print(model.layers[0].weight.shape)

torch.Size([30, 50])


In [32]:
print(model.layers[0].bias.shape)

torch.Size([30])


In [33]:
torch.manual_seed(42)
X = torch.rand((1, 50))
out = model(X)
print(out)

tensor([[ 0.1215, -0.0497, -0.1536]], grad_fn=<AddmmBackward0>)


AddmmBackward0: Refers to last operation done by the model. 'mm' stands for matrix multiplication followed by 'Add', i.e. Addition. This information will be used by the PyTorch when computing gradients during backpropagation

In the case of inferencing, we don't need to compute gradients, as there is only forward pass, no backward pass. In that case scenario, it's better to ignore the construction of computational graph for backpropagation, as it consumes additional memory.

In [ ]:
with torch.no_grad():     # This helps ignoring the computation of graph and keeping track of gradients, since only backpropagation is needed.
    out = model(X)
print(out)                # It can be observed that grad_fn=<AddmmBackward0> is not present in the o/p, due to no_grad()

tensor([[ 0.1215, -0.0497, -0.1536]])


Typically adding softmax or sigmoid is ignored in Pytorch and is done explicitly while computing the forward pass, as PyTorch automatically does the negative log likelyhood in a single class. To compute the softmax, again no_grad can be used.

In [35]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

tensor([[0.3843, 0.3238, 0.2919]])


## Creating a small toy dataset

In [36]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])
y_test = torch.tensor([0, 1])

NOTE: PyTorch requires that class labels start with label 0, and the largest class label value should not exceed the number of output nodes minus 1 (since Python index counting starts at zero). So, if we have class labels 0, 1, 2, 3, and 4, the neural network output layer should consist of five nodes.

## Defining a custom Dataset Class

In [37]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):              # Insturction to get exactly one record and the corresponding label
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):                         # Instruction to return total length of the dataset
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [38]:
print(len(train_ds))

5


## Instantiating data loaders

In [44]:
from torch.utils.data import DataLoader

torch.manual_seed(42)

train_loader = DataLoader(
    dataset = train_ds,                  # ToyDataset created earlier serves as input to the dataloader
    batch_size = 2,
    shuffle = True,                      # Whether or not to shuffle the data
    num_workers = 0                      # Number of background processes
)

test_loader = DataLoader(
    dataset = test_ds,
    batch_size = 2,
    shuffle = False,                     # Not necessary to shuffle the test dataset
    num_workers = 0
)

In [45]:
# Iterate over train dataset
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.5000,  2.6000]]) tensor([1, 0])
Batch 2: tensor([[-0.9000,  2.9000],
        [ 2.7000, -1.5000]]) tensor([0, 1])
Batch 3: tensor([[-1.2000,  3.1000]]) tensor([0])


In [ ]:
# Even with manual seeding, iterating second time gets rearranged batches, to prevent deep learning get the same batches in multiple epochs. But same cell getting executed
# after dataloader cell will give same output due to manual_seeding
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.7000, -1.5000],
        [-1.2000,  3.1000]]) tensor([1, 0])
Batch 2: tensor([[-0.9000,  2.9000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[ 2.3000, -1.1000]]) tensor([1])


In [47]:
# A training loader that drops the last batch. This is to prevent convergence problem while training due to unequal batch sizes.
train_loader = DataLoader(
    dataset = train_ds,
    batch_size = 2,
    shuffle = True,
    num_workers = 0,
    drop_last = True
)

for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-0.9000,  2.9000],
        [ 2.7000, -1.5000]]) tensor([0, 1])
Batch 2: tensor([[ 2.3000, -1.1000],
        [-0.5000,  2.6000]]) tensor([1, 0])


## Understanding Onehot encoding and Cross Entropy in PyTorch

In [8]:
def to_onehot(y, num_classes):
    y_onehot = torch.zeros(y.size(0), num_classes)
    y_onehot.scatter_(1, y.view(-1, 1).long(), 1).float()
    return y_onehot

y = torch.tensor([0, 1, 2, 2])

y_enc = to_onehot(y, num_classes=3)

print('one-hot encoding:\n', y_enc)

one-hot encoding:
 tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [0., 0., 1.]])
